In [1]:
import numpy as np

K_LIST = [1, 2, 3, 5, 8, 10]
N_SIM, SEED = 20000, 20260101


def step_success(p, c):
    """部署校验后单步"最终正确"的概率：一次成功，或失败被发现后重试成功。"""
    return p + (1 - p) * c * p


def simulate_task(p, k, c, rng):
    """返回一条 k 步任务的结局：correct / silent / detected。"""
    for _ in range(k):
        if rng.random() < p:
            continue                      # 该步一次成功
        if rng.random() < c:              # 失败被中间校验发现
            if rng.random() < p:          # 重试成功
                continue
            return "detected"             # 重试仍失败，任务中止
        return "silent"                   # 失败未被察觉，错误流入下游
    return "correct"


rng = np.random.default_rng(SEED)

for c in [0.0, 0.8]:
    print(f"--- 每步校验发现错误的概率 c={c} ---")
    print("p\\k  " + "".join(f"{k:>8}" for k in K_LIST))
    for p in [0.90, 0.95, 0.99]:
        cells = "".join(f"{step_success(p, c) ** k:>8.3f}" for k in K_LIST)
        print(f"{p:.2f} {cells}")

print("--- p=0.95、无校验时的衰减（ASCII 柱状） ---")
for k in K_LIST:
    v = 0.95 ** k
    print(f"k={k:>2}  {v:6.3f}  |{'#' * round(v * 50)}")

print("--- 蒙特卡洛核验：模拟 vs 解析 ---")
for p, k, c in [(0.95, 5, 0.0), (0.95, 5, 0.8), (0.90, 10, 0.8)]:
    est = sum(simulate_task(p, k, c, rng) == "correct"
              for _ in range(N_SIM)) / N_SIM
    se = (est * (1 - est) / N_SIM) ** 0.5
    print(f"p={p:.2f} k={k:>2} c={c:.1f} | 解析 {step_success(p, c) ** k:.4f}"
          f" | 模拟 {est:.4f} (SE {se:.4f})")

--- 每步校验发现错误的概率 c=0.0 ---
p\k         1       2       3       5       8      10
0.90    0.900   0.810   0.729   0.590   0.430   0.349
0.95    0.950   0.902   0.857   0.774   0.663   0.599
0.99    0.990   0.980   0.970   0.951   0.923   0.904
--- 每步校验发现错误的概率 c=0.8 ---
p\k         1       2       3       5       8      10
0.90    0.972   0.945   0.918   0.868   0.797   0.753
0.95    0.988   0.976   0.964   0.941   0.908   0.886
0.99    0.998   0.996   0.994   0.990   0.983   0.979
--- p=0.95、无校验时的衰减（ASCII 柱状） ---
k= 1   0.950  |################################################
k= 2   0.902  |#############################################
k= 3   0.857  |###########################################
k= 5   0.774  |#######################################
k= 8   0.663  |#################################
k=10   0.599  |##############################
--- 蒙特卡洛核验：模拟 vs 解析 ---
p=0.95 k= 5 c=0.0 | 解析 0.7738 | 模拟 0.7699 (SE 0.0030)
p=0.95 k= 5 c=0.8 | 解析 0.9414 | 模拟 0.9397 (SE 0.0017)
p=0.90 k=10 c=0.8

In [ ]:
# TOOLS = {
#     "run_r_script": lambda path: subprocess.run(["Rscript", path],
#                                                 capture_output=True, text=True).stdout,
#     "read_csv_head": lambda path: open(path).read()[:2000],
# }

# SYSTEM = """可用工具：{tools}
# 每轮输出严格一行 JSON：{{"thought": "...", "action": "工具名或 finish", "args": {{}}}}
# 收到 observation 后继续下一轮；决定结束时 action 取 finish。"""

# def react_loop(query, max_steps=8):
#     history = [{"role": "user", "content": query}]      # 历史即 h_t
#     for step in range(max_steps):
#         reply = call_llm(SYSTEM.format(tools=list(TOOLS)), history)  # 采样 thought + action
#         call = json.loads(reply)
#         history.append({"role": "assistant", "content": reply})
#         if call["action"] == "finish":                  # 终止条件
#             return call["args"].get("answer")
#         try:
#             obs = TOOLS[call["action"]](**call["args"]) # 环境返回观测
#         except Exception as exc:
#             obs = f"ERROR: {exc}"                       # 错误也作为观测，交给模型修正
#         history.append({"role": "user", "content": f"observation: {obs}"})
#     return None                                         # 超出步数上限，按失败处理

In [2]:
import numpy as np

P_OK, P_FMT, P_SEM_VIS, P_SEM_INV = 0.75, 0.15, 0.08, 0.02
MAX_ATTEMPTS, N_SIM, SEED = 3, 20000, 123
OUTCOMES = ["success", "detected", "silent"]


def draw_call(rng):
    u = rng.random()
    if u < P_OK:
        return "ok"
    if u < P_OK + P_FMT:
        return "format"
    if u < P_OK + P_FMT + P_SEM_VIS:
        return "sem_vis"
    return "sem_inv"


def run_task(strategy, rng):
    for i in range(MAX_ATTEMPTS):
        last = i == MAX_ATTEMPTS - 1
        o = draw_call(rng)
        if o == "ok":
            return "success"
        if o == "format":
            # 无校验时格式错误在运行时崩溃，可察觉；有 Schema 时拦截重试
            if strategy != "none" and not last:
                continue
            return "detected"
        if o == "sem_vis":
            if strategy == "schema+runtime" and not last:
                continue                    # 运行时断言发现，重试
            if strategy == "schema":
                return "silent"             # Schema 对语义盲，静默通过
            return "detected"               # 无校验：运行时报错
        return "silent"                     # sem_inv：三种策略都静默
    return "detected"


rng = np.random.default_rng(SEED)
print(f"单次尝试分布：正确 {P_OK} / 格式错误 {P_FMT} / "
      f"可发现语义错误 {P_SEM_VIS} / 不可发现语义错误 {P_SEM_INV}")
print(f"最多尝试 {MAX_ATTEMPTS} 次，n_sim={N_SIM}，seed={SEED}")
print(f"{'策略':<16}{'成功':>8}{'被察觉失败':>12}{'静默错误':>10}"
      f"{'静默占完成比':>13}")
for strategy, label in [("none", "A 无校验"),
                        ("schema", "B 仅Schema"),
                        ("schema+runtime", "C Schema+断言")]:
    res = [run_task(strategy, rng) for _ in range(N_SIM)]
    n = len(res)
    s = {k: res.count(k) / n for k in OUTCOMES}
    share = s["silent"] / (1 - s["detected"])
    print(f"{label:<16}{s['success']:>8.4f}{s['detected']:>12.4f}"
          f"{s['silent']:>10.4f}{share:>13.4f}")

单次尝试分布：正确 0.75 / 格式错误 0.15 / 可发现语义错误 0.08 / 不可发现语义错误 0.02
最多尝试 3 次，n_sim=20000，seed=123
策略                    成功       被察觉失败      静默错误       静默占完成比
A 无校验             0.7557      0.2250    0.0193       0.0250
B 仅Schema         0.8800      0.0031    0.1169       0.1173
C Schema+断言       0.9639      0.0112    0.0249       0.0252


In [ ]:
# from voyager import Voyager

# azure_login = {
#     'client_id': 'YOUR_CLIENT_ID',
#     'redirect_url': 'https://127.0.0.1/auth-response',
#     'secret_value': '[OPTIONAL] YOUR_SECRET_VALUE',
#     'version': 'fabric-loader-0.14.18-1.19',
# }
# voyager = Voyager(azure_login=azure_login, openai_api_key='YOUR_API_KEY')
# voyager.learn()

# # 从检查点恢复训练
# voyager = Voyager(azure_login=azure_login, openai_api_key='YOUR_API_KEY',
#                   ckpt_dir="YOUR_CKPT_DIR", resume=True)

# # 加载已学技能库解决新任务
# voyager = Voyager(azure_login=azure_login, openai_api_key='YOUR_API_KEY',
#                   skill_library_dir="./skill_library/trial1",
#                   ckpt_dir="YOUR_CKPT_DIR", resume=False)
# sub_goals = voyager.decompose_task(task="Craft a diamond pickaxe")
# voyager.inference(sub_goals=sub_goals)

In [ ]:
# """蒙特卡洛假设检验自动验证脚本：估计两样本 t 检验的第一类错误率、功效与覆盖率。"""
# import numpy as np
# from scipy import stats
# import json
# import sys


# def simulate_ttest(alpha=0.05, n1=30, n2=30,
#                    effect_size=0.0, n_sim=10000, seed=42):
#     """模拟两样本 t 检验的第一类错误率与功效。

#     参数
#     ----
#     alpha        : 名义显著性水平
#     n1, n2       : 两组样本量
#     effect_size  : 0 表示 H0 为真；非 0 表示第二组均值偏移该量
#     n_sim        : 重复次数，越大蒙特卡洛误差越小
#     seed         : 随机种子，保证结果可复现
#     """
#     np.random.seed(seed)
#     false_positive = 0
#     true_positive = 0
#     coverage_count = 0

#     for _ in range(n_sim):
#         if effect_size == 0:
#             # H0 为真：两组来自同一正态分布
#             x = np.random.normal(0, 1, n1)
#             y = np.random.normal(0, 1, n2)
#         else:
#             # H1 为真：第二组有偏移
#             x = np.random.normal(0, 1, n1)
#             y = np.random.normal(effect_size, 1, n2)

#         t_stat, p_value = stats.ttest_ind(x, y)

#         # 记录第一类错误（仅 H0 为真时计数）
#         if effect_size == 0 and p_value < alpha:
#             false_positive += 1

#         # 记录功效（仅 H1 为真时计数）
#         if effect_size != 0 and p_value < alpha:
#             true_positive += 1

#         # 记录覆盖率：用 t 分布构造 1-alpha 置信区间
#         # 注意：np.var 默认 ddof=0，与 ttest_ind 内部的合并方差口径略有差异
#         se = np.sqrt(x.var() / n1 + y.var() / n2)
#         df = n1 + n2 - 2
#         t_crit = stats.t.ppf(1 - alpha / 2, df)
#         diff = x.mean() - y.mean()
#         ci_lower = diff - t_crit * se
#         ci_upper = diff + t_crit * se
#         if ci_lower <= effect_size <= ci_upper:
#             coverage_count += 1

#     results = {
#         "simulated_alpha": false_positive / n_sim if effect_size == 0 else None,
#         "simulated_power": true_positive / n_sim if effect_size != 0 else None,
#         "coverage_rate": coverage_count / n_sim,
#         "nominal_alpha": alpha,
#         "n_sim": n_sim,
#         "seed": seed,
#         "environment": {
#             "numpy": np.__version__,
#             "scipy": stats.__name__,
#         },
#     }
#     return results


# if __name__ == "__main__":
#     import argparse

#     parser = argparse.ArgumentParser()
#     parser.add_argument("--test", default="ttest")
#     parser.add_argument("--alpha", type=float, default=0.05)
#     parser.add_argument("--n1", type=int, default=30)
#     parser.add_argument("--n2", type=int, default=30)
#     parser.add_argument("--effect-size", type=float, default=0.0)
#     parser.add_argument("--n-sim", type=int, default=10000)
#     parser.add_argument("--output", default="results.json")
#     args = parser.parse_args()

#     results = simulate_ttest(
#         alpha=args.alpha, n1=args.n1, n2=args.n2,
#         effect_size=args.effect_size, n_sim=args.n_sim,
#     )

#     with open(args.output, "w") as f:
#         json.dump(results, f, indent=2)

#     print(json.dumps(results, indent=2))

In [ ]:
# """功效分析：解析法求样本量 + 蒙特卡洛复核 + 功效曲线。"""
# import numpy as np
# from scipy import stats
# from statsmodels.stats.power import TTestIndPower, FTestAnovaPower


# def required_n(d, alpha=0.05, power=0.80, ratio=1.0):
#     """两独立样本 t 检验：给定 Cohen's d 反解每组样本量（解析解，向上取整）。"""
#     analysis = TTestIndPower()
#     n = analysis.solve_power(effect_size=d, alpha=alpha, power=power, ratio=ratio)
#     return int(np.ceil(n))


# def mc_power(d, n, alpha=0.05, n_sim=5000, seed=20260101):
#     """蒙特卡洛复核：用模拟估计给定 n 与 d 下的实际功效。"""
#     rng = np.random.default_rng(seed)          # 显式 RNG，避免全局种子污染
#     reject = 0
#     for _ in range(n_sim):
#         x = rng.normal(0.0, 1.0, n)
#         y = rng.normal(d, 1.0, n)              # 标准化设定下均值差即 d
#         _, p = stats.ttest_ind(x, y)
#         reject += (p < alpha)
#     return reject / n_sim


# def power_curve(d, ns, alpha=0.05, n_sim=2000, seed=20260101):
#     """在一组候选样本量上计算功效，用于绘图。"""
#     return [(n, mc_power(d, n, alpha, n_sim, seed)) for n in ns]

In [ ]:
# import numpy as np


# def split_conformal(f, X_cal, y_cal, alpha=0.1):
#     """返回校准分位数 q̂：使区间 [f(x)-q̂, f(x)+q̂] 达到 1-alpha 边际覆盖。"""
#     scores = np.abs(y_cal - f.predict(X_cal))            # conformity score
#     n_cal = len(scores)
#     level = np.ceil((n_cal + 1) * (1 - alpha)) / n_cal   # 有限样本修正
#     return np.quantile(scores, level, method="higher")


# def evaluate(f, qhat, X_test, y_test, alpha=0.1):
#     """在独立测试集上评估经验覆盖率与平均宽度。"""
#     pred = f.predict(X_test)
#     lower, upper = pred - qhat, pred + qhat
#     covered = (y_test >= lower) & (y_test <= upper)
#     return {
#         "nominal": 1 - alpha,
#         "empirical_coverage": float(covered.mean()),
#         "mean_width": float((upper - lower).mean()),
#         "n_test": len(y_test),
#     }

In [ ]:
# import urllib.request
# import urllib.parse
# import xml.etree.ElementTree as ET


# def search_arxiv(topic, max_results=20):
#     """调用 arXiv Atom API 检索按提交时间倒序排列的论文。"""
#     base_url = "http://export.arxiv.org/api/query?"
#     query = (f'search_query=all:"{topic}"'
#              f'&sortBy=submittedDate&sortOrder=descending&max_results={max_results}')
#     url = base_url + urllib.parse.quote(query)
#     with urllib.request.urlopen(url) as response:      # 生产环境需加超时与重试
#         xml_data = response.read().decode("utf-8")

#     root = ET.fromstring(xml_data)
#     ns = {"atom": "http://www.w3.org/2005/Atom"}
#     papers = []
#     for entry in root.findall("atom:entry", ns):
#         papers.append({
#             "title": entry.find("atom:title", ns).text.strip(),
#             "authors": [a.find("atom:name", ns).text
#                         for a in entry.findall("atom:author", ns)],
#             "abstract": entry.find("atom:summary", ns).text.strip(),
#             "date": entry.find("atom:published", ns).text[:10],
#             "link": entry.find("atom:id", ns).text,
#         })
#     return papers


# if __name__ == "__main__":
#     import sys
#     topic = sys.argv[1] if len(sys.argv) > 1 else "machine learning"
#     for p in search_arxiv(topic):
#         print(f"[{p['date']}] {p['title']}")
#         print(f"  Authors: {', '.join(p['authors'][:3])}")
#         print(f"  Link: {p['link']}")
#         print()

In [3]:
import numpy as np
from scipy import stats

M, M1, ALPHA, Q, N_SIM, SEED = 100, 5, 0.05, 0.05, 5000, 20260101


def bh_select(pvals, q):
    """Benjamini-Hochberg 步进法：返回被判为相关的工具掩码。"""
    order = np.argsort(pvals)
    thresh = q * np.arange(1, len(pvals) + 1) / len(pvals)
    passed = np.where(pvals[order] <= thresh)[0]
    sel = np.zeros(len(pvals), dtype=bool)
    if passed.size:
        sel[order[: passed[-1] + 1]] = True
    return sel


def run_regime(mu):
    """在给定效应量 mu 下，估计两种注入规则的召回与假阳性。"""
    rng = np.random.default_rng(SEED)
    true_mask = np.array([True] * M1 + [False] * (M - M1))
    rows = {"A 不校正": [], "B BH(q=0.05)": []}
    for _ in range(N_SIM):
        z = rng.normal(0.0, 1.0, M)
        z[true_mask] += mu
        pvals = 1.0 - stats.norm.cdf(z)          # 单侧 p 值
        for rule, sel in [("A 不校正", pvals < ALPHA),
                          ("B BH(q=0.05)", bh_select(pvals, Q))]:
            V = int((sel & ~true_mask).sum())    # 假阳性：无关工具被注入
            T = int((sel & true_mask).sum())     # 选真：真相关工具被注入
            rows[rule].append((sel.sum(), T, V))
    print(f"mu={mu}：不校正时至少注入一个无关工具的概率（解析）"
          f"= {1 - (1 - ALPHA) ** (M - M1):.4f}")
    for rule, lst in rows.items():
        arr = np.array(lst, dtype=float)
        fdr = (arr[:, 2] / np.maximum(arr[:, 0], 1)).mean()
        miss = 1 - arr[:, 1].mean() / M1
        print(f"  {rule}: 选中 R={arr[:, 0].mean():.2f}  "
              f"选真={arr[:, 1].mean():.2f}  假阳性 V={arr[:, 2].mean():.2f}  "
              f"经验FDR={fdr:.3f}  真工具漏检率={miss:.3f}")


for mu in [3.0, 4.0]:
    run_regime(mu)

mu=3.0：不校正时至少注入一个无关工具的概率（解析）= 0.9923
  A 不校正: 选中 R=9.37  选真=4.55  假阳性 V=4.81  经验FDR=0.490  真工具漏检率=0.089
  B BH(q=0.05): 选中 R=2.76  选真=2.57  假阳性 V=0.19  经验FDR=0.047  真工具漏检率=0.486
mu=4.0：不校正时至少注入一个无关工具的概率（解析）= 0.9923
  A 不校正: 选中 R=9.76  选真=4.95  假阳性 V=4.81  经验FDR=0.468  真工具漏检率=0.010
  B BH(q=0.05): 选中 R=4.65  选真=4.39  假阳性 V=0.27  经验FDR=0.046  真工具漏检率=0.123
